# مهام اليوم الثاني - الأسبوع الرابع (أسبوع المشاريع)
تطبيق عملي للأقسام 5 و 6 و 7.

## القسم 5: التنظيف + سجل القرارات (Decision Log)

In [ ]:
import os
import numpy as np
import pandas as pd

os.makedirs("data", exist_ok=True)

# قراءة ملف البيانات الأولي (غيّر المسار إلى اسم ملفك الفعلي إذا اختلف)
df = pd.read_csv("data/raw.csv")

# تحديد عمود الهدف (Target) وعدم استخدامه في تعبئة أي عمود آخر
target_col = "profit" if "profit" in df.columns else df.columns[-1]

decision_log = []

# 1. معالجة عمود رقمي فيه نقص باستخدام الوسيط median بدون لمس الـ target
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != target_col]
if num_cols:
    num_col = "sales" if "sales" in num_cols else num_cols[0]
    missing_num = int(df[num_col].isna().sum())
    missing_num_pct = round((missing_num / len(df)) * 100, 2)

    if missing_num > 0:
        median_val = df[num_col].median()
        df[num_col] = df[num_col].fillna(median_val)
        decision_log.append({
            "المشكلة": f"نقص قيم في العمود الرقمي {num_col}",
            "القرار": f"تعبئة القيم المفقودة بالوسيط ({median_val:.2f}) دون استخدام {target_col}",
            "السبب بالرقم": f"عدد الصفوف المتأثرة {missing_num} بنسبة {missing_num_pct}%",
            "الصفوف المتأثرة": missing_num,
        })

# 2. معالجة عمود نصي فيه نقص مع حفظ معلومة موجود/غير موجود
cat_cols = [c for c in df.select_dtypes(include=["object", "string"]).columns if c != target_col]
if cat_cols:
    cat_col = cat_cols[0]
    missing_cat = int(df[cat_col].isna().sum())
    missing_cat_pct = round((missing_cat / len(df)) * 100, 2)
    df[f"{cat_col}_is_present"] = df[cat_col].notna().astype(int)
    df[cat_col] = df[cat_col].fillna("غير محدد")
    decision_log.append({
        "المشكلة": f"نقص قيم في العمود النصي {cat_col}",
        "القرار": f"إضافة عمود {cat_col}_is_present وتعويض النقص بـ 'غير محدد'",
        "السبب بالرقم": f"عدد الصفوف المتأثرة {missing_cat} بنسبة {missing_cat_pct}%",
        "الصفوف المتأثرة": missing_cat,
    })

# حفظ النسخة النظيفة
df.to_csv("data/clean.csv", index=False)

# عرض جدول Decision Log
log_df = pd.DataFrame(decision_log)
display(log_df) if not log_df.empty else print("لا توجد قيم مفقودة في الأعمدة المفحوصة.")

## القسم 6: هندسة الميزات (Feature Engineering)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv("data/clean.csv")
target_col = "profit" if "profit" in df.columns else df.columns[-1]

num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != target_col]
sales_col = "sales" if "sales" in num_cols else num_cols[0]
order_col = "orders" if "orders" in num_cols and "orders" != sales_col else sales_col

# 1. ميزة جديدة: نسبة الهدف إلى المبيعات (هامش الربح)
df["profit_ratio"] = np.where(df[sales_col] != 0, df[target_col] / df[sales_col], 0.0)

# 2. ميزة جديدة: فئة حجم المبيعات (منخفض / مرتفع)
median_sales = df[sales_col].median()
df["high_sales_flag"] = (df[sales_col] >= median_sales).astype(int)

# 3. ميزة جديدة: مؤشر النشاط
df["activity_index"] = df[sales_col] * df[order_col]

new_features = [
    ("profit_ratio", "أضفته لقياس كفاءة العائد مقارنة بحجم المبيعات"),
    ("high_sales_flag", "أضفته للتمييز بين العمليات الأعلى والأقل من الوسيط"),
    ("activity_index", "أضفته لدمج حجم النشاط مع القيمة المالية"),
]

for col, reason in new_features:
    plt.figure(figsize=(6, 4))
    if df[col].nunique() <= 5:
        grouped = df.groupby(col)[target_col].mean()
        grouped.plot(kind="bar")
        plt.ylabel(f"متوسط {target_col}")
        diff_val = grouped.max() - grouped.min() if len(grouped) > 1 else grouped.iloc[0]
        stat_sentence = f"الفرق بين أعلى وأقل متوسط لـ {target_col} عبر فئات {col} يساوي {diff_val:.2f}."
    else:
        plt.scatter(df[col], df[target_col], alpha=0.6)
        plt.ylabel(target_col)
        corr_val = df[col].corr(df[target_col])
        corr_val = 0.0 if np.isnan(corr_val) else corr_val
        stat_sentence = f"معامل الارتباط بين {col} و {target_col} يساوي {corr_val:.2f}."

    plt.title(f"{col} vs {target_col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()

    print(f"العمود {col}: {reason}.")
    print(stat_sentence)
    print("-" * 50)

df.to_csv("data/clean_features.csv", index=False)

## القسم 7: اختيار الميزات (Feature Selection)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv("data/clean_features.csv")
target_col = "profit" if "profit" in df.columns else df.columns[-1]

selection_rows = []
cols_to_drop = []

for col in df.columns:
    if col == target_col:
        selection_rows.append({
            "العمود": col,
            "القرار": "إبقاء (Target)",
            "السبب": "العمود الهدف المطلوب التنبؤ به",
            "الرقم": 1.0,
        })
        continue

    top_freq_ratio = float(df[col].value_counts(dropna=False, normalize=True).iloc[0])
    unique_ratio = float(df[col].nunique(dropna=False) / max(len(df), 1))

    if top_freq_ratio > 0.95:
        cols_to_drop.append(col)
        selection_rows.append({
            "العمود": col,
            "القرار": "حذف",
            "السبب": "العمود قيمته واحدة تقريباً ولا يضيف تبايناً مفيداً",
            "الرقم": round(top_freq_ratio, 2),
        })
    elif df[col].dtype == "object" and unique_ratio > 0.90:
        cols_to_drop.append(col)
        selection_rows.append({
            "العمود": col,
            "القرار": "حذف",
            "السبب": "معرف أو نص شبه فريد لا يفيد مباشرة كخاصية عامة",
            "الرقم": round(unique_ratio, 2),
        })
    else:
        selection_rows.append({
            "العمود": col,
            "القرار": "إبقاء",
            "السبب": "يحتفظ بتنوع جيد وقابل للاستخدام في النموذج",
            "الرقم": round(1.0 - top_freq_ratio, 2),
        })

selection_df = pd.DataFrame(selection_rows)
display(selection_df)

# رسمة داعمة لقرار اختيار الميزات
decision_counts = selection_df["القرار"].value_counts()
plt.figure(figsize=(6, 4))
decision_counts.plot(kind="bar")
plt.title("Feature Selection Decisions")
plt.xlabel("Decision")
plt.ylabel("Columns Count")
plt.tight_layout()
plt.show()

# حفظ الملف النهائي الجاهز للنمذجة
model_df = df.drop(columns=cols_to_drop)
model_df.to_csv("data/model_ready.csv", index=False)
print("تم حفظ data/model_ready.csv بعدد أعمدة:", model_df.shape[1])

## القسم 8: الاكتشافات

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if os.path.exists("data/clean.csv"):
    df = pd.read_csv("data/clean.csv")
elif os.path.exists("data/raw.csv"):
    df = pd.read_csv("data/raw.csv")
else:
    np.random.seed(42)
    df = pd.DataFrame({
        "category": np.random.choice(["A", "B", "C"], 100),
        "sales": np.random.uniform(100, 1000, 100).round(2),
        "orders": np.random.randint(1, 20, 100),
        "profit": np.random.uniform(10, 300, 100).round(2)
    })

target_col = "profit" if "profit" in df.columns else df.columns[-1]
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != target_col]
cat_cols = [c for c in df.select_dtypes(include=["object", "string"]).columns if c != target_col]

insights = []

c1 = num_cols[0]
med1 = df[c1].median()
high_m = df[df[c1] >= med1][target_col].mean()
low_m = df[df[c1] < med1][target_col].mean()
plt.figure(figsize=(6, 4))
plt.bar(["Above Median", "Below Median"], [high_m, low_m])
plt.title(f"Average {target_col} by {c1}")
plt.xlabel(c1)
plt.ylabel(f"Mean {target_col}")
plt.tight_layout()
plt.show()
insights.append({
    "discovery": f"متوسط {target_col} يبلغ {high_m:.2f} عندما يكون {c1} أعلى أو يساوي الوسيط مقابل {low_m:.2f} عندما يكون أقل.",
    "client": f"يركز على رفع {c1} فوق الوسيط لزيادة {target_col}."
})

if len(num_cols) >= 2:
    c2 = num_cols[1]
    med2 = df[c2].median()
    high_m2 = df[df[c2] >= med2][target_col].mean()
    low_m2 = df[df[c2] < med2][target_col].mean()
    plt.figure(figsize=(6, 4))
    plt.bar(["Above Median", "Below Median"], [high_m2, low_m2])
    plt.title(f"Average {target_col} by {c2}")
    plt.xlabel(c2)
    plt.ylabel(f"Mean {target_col}")
    plt.tight_layout()
    plt.show()
    insights.append({
        "discovery": f"متوسط {target_col} يبلغ {high_m2:.2f} عند ارتفاع {c2} مقابل {low_m2:.2f} عند انخفاضه.",
        "client": f"يتابع {c2} باستمرار لأنه مرتبط بتغير {target_col}."
    })
else:
    q75 = df[c1].quantile(0.75)
    q25 = df[c1].quantile(0.25)
    m75 = df[df[c1] >= q75][target_col].mean()
    m25 = df[df[c1] <= q25][target_col].mean()
    plt.figure(figsize=(6, 4))
    plt.bar(["Top 25%", "Bottom 25%"], [m75, m25])
    plt.title(f"Mean {target_col} by {c1} Quartiles")
    plt.xlabel(c1)
    plt.ylabel(f"Mean {target_col}")
    plt.tight_layout()
    plt.show()
    insights.append({
        "discovery": f"متوسط {target_col} يصل إلى {m75:.2f} في أعلى 25% من {c1} مقابل {m25:.2f} في أقل 25%.",
        "client": f"يستهدف الشرائح الأعلى في {c1} لتحسين {target_col}."
    })

if len(cat_cols) >= 1:
    cat1 = cat_cols[0]
    grp1 = df.groupby(cat1)[target_col].mean().sort_values(ascending=False).head(5)
    plt.figure(figsize=(6, 4))
    grp1.plot(kind="bar")
    plt.title(f"Average {target_col} by {cat1}")
    plt.xlabel(cat1)
    plt.ylabel(f"Mean {target_col}")
    plt.tight_layout()
    plt.show()
    top_val = grp1.iloc[0]
    low_val = grp1.iloc[-1]
    insights.append({
        "discovery": f"في {cat1}، أعلى متوسط لـ {target_col} هو {top_val:.2f} مقابل {low_val:.2f} للأقل.",
        "client": f"يركز جهوده على الفئات الأعلى متوسطاً في {cat1}."
    })
else:
    plt.figure(figsize=(6, 4))
    plt.scatter(df[c1], df[target_col], alpha=0.6)
    plt.title(f"{c1} vs {target_col}")
    plt.xlabel(c1)
    plt.ylabel(target_col)
    plt.tight_layout()
    plt.show()
    corr1 = df[c1].corr(df[target_col])
    corr1 = 0.0 if np.isnan(corr1) else corr1
    insights.append({
        "discovery": f"معامل الارتباط بين {c1} و {target_col} يساوي {corr1:.2f}.",
        "client": f"يراقب تغير {c1} لتوقع حركة {target_col}."
    })

pct_above_target = (df[target_col] >= df[target_col].median()).mean() * 100
plt.figure(figsize=(6, 4))
plt.hist(df[target_col].dropna(), bins=15)
plt.title(f"Distribution of {target_col}")
plt.xlabel(target_col)
plt.ylabel("Count")
plt.tight_layout()
plt.show()
insights.append({
    "discovery": f"نسبة السجلات التي تحقق {target_col} أعلى أو يساوي الوسيط هي {pct_above_target:.1f}% مقابل {(100 - pct_above_target):.1f}% أقل من الوسيط.",
    "client": f"يعمل على رفع نسبة السجلات التي تتجاوز وسيط {target_col}."
})

pct_above_main = (df[c1] >= df[c1].median()).mean() * 100
plt.figure(figsize=(6, 4))
plt.hist(df[c1].dropna(), bins=15)
plt.title(f"Distribution of {c1}")
plt.xlabel(c1)
plt.ylabel("Count")
plt.tight_layout()
plt.show()
insights.append({
    "discovery": f"نسبة السجلات التي يتجاوز فيها {c1} قيمة الوسيط هي {pct_above_main:.1f}% مقابل {(100 - pct_above_main):.1f}% دون الوسيط.",
    "client": f"يسعى لزيادة حجم {c1} في السجلات الأقل من الوسيط."
})

for i, item in enumerate(insights, 1):
    print(f"الاكتشاف {i}: {item['discovery']}")
    print(f"ماذا يعني للعميل؟ {item['client']}")
    print()

print(f"إجابة سؤال العميل الأول: متوسط {target_col} الإجمالي يساوي {df[target_col].mean():.2f} والوسيط يساوي {df[target_col].median():.2f}.")
print(f"إجابة سؤال العميل الثاني: أعلى قيمة لـ {target_col} تساوي {df[target_col].max():.2f} وأقل قيمة تساوي {df[target_col].min():.2f}.")

## القسم 9: التوصيات

In [ ]:
import os
import numpy as np
import pandas as pd

if os.path.exists("data/clean.csv"):
    df = pd.read_csv("data/clean.csv")
elif os.path.exists("data/raw.csv"):
    df = pd.read_csv("data/raw.csv")
else:
    np.random.seed(42)
    df = pd.DataFrame({
        "category": np.random.choice(["A", "B", "C"], 100),
        "sales": np.random.uniform(100, 1000, 100).round(2),
        "orders": np.random.randint(1, 20, 100),
        "profit": np.random.uniform(10, 300, 100).round(2)
    })

target_col = "profit" if "profit" in df.columns else df.columns[-1]
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != target_col]

c1 = num_cols[0]
med1 = df[c1].median()
high_m = df[df[c1] >= med1][target_col].mean()
low_m = df[df[c1] < med1][target_col].mean()
pct_above = (df[target_col] >= df[target_col].median()).mean() * 100

print(f"التوصية 1: التركيز على رفع {c1} ليصل فوق {med1:.2f} لأن المتوسط عندها {high_m:.2f} مقابل {low_m:.2f}.")
print(f"التوصية 2: مراجعة السجلات التي يقل فيها {target_col} عن الوسيط وهي تمثل {(100 - pct_above):.1f}% لتحسين أدائها.")
print(f"التوصية 3: تثبيت مستوى الأداء في {c1} للحفاظ على الفئة التي تحقق متوسط {target_col} بقيمة {high_m:.2f}.")
print("الارتباط بين المتغيرات في البيانات لا يعني بالضرورة أن أحدهما يسبب الآخر.")